# AtendeAI — Fase 4.1: dois ajustes, medidos antes de adotar
Continua a Fase 4. **Uso de IA:** ver `docs/uso_de_ia.md`. Este notebook **não substitui** o 04: as evidências da Fase 4 ficam como estão.

## Por que existe
A Fase 4 revelou duas fragilidades, ambas registradas como limitações:
1. **RAG:** a reescrita de consulta corrigiu 3 falhas, mas piorou 2 acertos (73% de hit@1 antes e depois) e variou entre execuções.
2. **Text-to-query:** em "Quantos pedidos eu já fiz e quanto gastei?", o modelo acrescentou sozinho `WHERE status != 'cancelado'` e a resposta saiu errada (7 pedidos em vez de 8).

## Regra do jogo (definida **antes** de ver os resultados)
Cada ajuste só é adotado se passar no critério abaixo, **com `N_REP` repetições** (o LLM varia entre chamadas). Se não passar, o resultado negativo entra no relatório.

| Ajuste | Critério de adoção |
|---|---|
| **A. Prompt SQL v2** (regra "não adicione filtros que a pergunta não pediu") | Acerta mais nas 3 perguntas de agregação do que a v1, **e** não acerta menos nas 2 perguntas de controle |
| **B. Fusão (consulta original + reescrita, por RRF)** | hit@1 e hit@3 médios **não menores** que os da consulta direta, **e** nenhuma pergunta que a consulta direta acertava em hit@1 é perdida em 2 ou mais repetições |

**Limites:** amostra pequena (11 perguntas de recuperação, 5 de SQL), sem teste estatístico. Os resultados indicam tendência, não prova.

> **Custo:** cerca de 70 chamadas ao LLM com `N_REP = 3`. Só faz sentido com `LLM_MODE=openai`.

In [ ]:
import os, re, sys, sqlite3, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

import pandas as pd
pd.set_option("display.max_colwidth", 100)

from src.llm_factory import get_llm
from src.sql_agente import criar_tool_sql
from src.sql_seguro import executar_isolado
from src.recuperacao import Recuperador

llm = get_llm()
MODO_DUMMY = os.getenv("LLM_MODE", "dummy").lower() == "dummy"
N_REP = 3
print("Modo:", os.getenv("LLM_MODE", "dummy"), "|", type(llm).__name__, "| repetições:", N_REP)
if MODO_DUMMY:
    print("ATENÇÃO: no modo dummy os dois ajustes não têm efeito; use o LLM real.")

con = sqlite3.connect(ROOT / "data" / "loja.db")
CLIENTE = con.execute("SELECT cliente_id FROM pedidos WHERE codigo_rastreio IS NOT NULL "
                      "GROUP BY cliente_id ORDER BY COUNT(*) DESC LIMIT 1").fetchone()[0]
PEDIDO, STATUS, CODIGO = con.execute("SELECT id, status, codigo_rastreio FROM pedidos "
                                     "WHERE cliente_id=? AND codigo_rastreio IS NOT NULL ORDER BY id LIMIT 1", (CLIENTE,)).fetchone()
con.close()
print(f"Cliente {CLIENTE} | pedido {PEDIDO} ({STATUS}, {CODIGO})")

## A. Text-to-query: prompt v1 × v2
O **gabarito vem do banco** (SQL escrito à mão, executado na mesma cópia isolada), não do LLM.

In [ ]:
_, linha = executar_isolado("SELECT COUNT(*), SUM(valor_total), AVG(valor_total) FROM pedidos", CLIENTE)
N_PED, SOMA, MEDIA = linha[0]
ID_MAIS_CARO = executar_isolado("SELECT id FROM pedidos ORDER BY valor_total DESC, id LIMIT 1", CLIENTE)[1][0][0]
print(f"Gabarito do cliente {CLIENTE}: {N_PED} pedidos | soma {SOMA:.2f} | média {MEDIA:.2f} | mais caro: pedido {ID_MAIS_CARO}")

def numeros(texto):
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", texto.split("(SQL executado")[0])]
def tem(texto, alvo, tol=0.01):
    return any(abs(n - alvo) <= tol for n in numeros(texto))

TESTES = [  # (grupo, nome, pergunta, verificador)
    ("agregação", "contagem e soma", "Quantos pedidos eu já fiz e quanto gastei no total?", lambda t: tem(t, N_PED, 0.001) and tem(t, SOMA)),
    ("agregação", "valor médio",     "Qual o valor médio dos meus pedidos?",                lambda t: tem(t, MEDIA)),
    ("agregação", "pedido mais caro", "Qual foi o meu pedido mais caro?",                    lambda t: tem(t, ID_MAIS_CARO, 0.001)),
    ("controle",  "status",          f"Qual o status do pedido {PEDIDO}?",                   lambda t: STATUS in t),
    ("controle",  "rastreio",        f"Qual o código de rastreio do pedido {PEDIDO}?",       lambda t: CODIGO in t),
]

In [ ]:
registros = []
for versao in ("v1", "v2"):
    tool = criar_tool_sql(llm, CLIENTE, versao=versao)
    for rep in range(1, N_REP + 1):
        for grupo, nome, pergunta, ok in TESTES:
            saida = tool.invoke(pergunta)
            sql = (re.search(r"\(SQL executado: (.*)\)\s*$", saida, re.S) or [None, ""])[1]
            registros.append({"versão": versao, "rep": rep, "grupo": grupo, "teste": nome, "acertou": bool(ok(saida)), "SQL": sql.replace("\n", " ")})
    print(f"{versao} concluída")
sql_df = pd.DataFrame(registros)
tabela_sql = sql_df.pivot_table(index=["grupo", "teste"], columns="versão", values="acertou", aggfunc="sum").astype(int)
tabela_sql.columns = [f"{c}: acertos de {N_REP}" for c in tabela_sql.columns]
tabela_sql

In [ ]:
agg = sql_df[sql_df["grupo"] == "agregação"].groupby("versão")["acertou"].sum()
ctl = sql_df[sql_df["grupo"] == "controle"].groupby("versão")["acertou"].sum()
adotar_v2 = bool(agg["v2"] > agg["v1"] and ctl["v2"] >= ctl["v1"])
print(f"Agregação: v1 = {agg['v1']} | v2 = {agg['v2']} acertos de {3 * N_REP}")
print(f"Controle : v1 = {ctl['v1']} | v2 = {ctl['v2']} acertos de {2 * N_REP}")
print("DECISÃO (regra definida antes):", "ADOTAR a v2" if adotar_v2 else "NÃO ADOTAR a v2; manter a v1 e registrar o resultado")
print()
print("Erros (SQL gerado):")
sql_df[~sql_df["acertou"]][["versão", "rep", "teste", "SQL"]]

## B. Recuperação: consulta direta × reescrita × fusão
**Desenho pareado:** em cada repetição, a reescrita de cada pergunta é gerada **uma vez** e usada pelas duas estratégias que dependem dela (reescrita pura e fusão), então a comparação entre elas é justa e custa 1 chamada por pergunta e repetição. A consulta direta é determinística (sem LLM).

In [ ]:
CASOS = {
    "Quero meu dinheiro de volta, como funciona?":     {"06_reembolso_e_estorno.md", "01_troca_e_devolucao.md"},
    "Posso desistir da compra depois que chegou?":     {"01_troca_e_devolucao.md"},
    "Comprei o tamanho errado, dá pra trocar?":        {"01_troca_e_devolucao.md"},
    "Quero devolver o tênis porque não gostei":        {"01_troca_e_devolucao.md"},
    "O produto veio quebrado":                         {"05_garantia_e_defeitos.md"},
    "Quanto tempo tenho para reclamar de defeito?":    {"05_garantia_e_defeitos.md"},
    "Quanto demora pra chegar no Nordeste?":           {"03_frete_e_cep.md", "02_prazos_de_entrega.md"},
    "Paguei no boleto e o pedido não andou":           {"02_prazos_de_entrega.md"},
    "Meu pedido atrasou, o que eu faço?":              {"02_prazos_de_entrega.md"},
    "Quero cancelar uma compra que já saiu da loja":   {"04_cancelamento.md"},
    "Como funciona o estorno no cartão?":              {"06_reembolso_e_estorno.md"},
}
rec = Recuperador(llm=llm, k=3)

def acertos(resultados, esperado):
    fontes = [r["fonte"] for r in resultados]
    return fontes[0] in esperado, any(f in esperado for f in fontes)

linhas = []
for rep in range(1, N_REP + 1):
    for pergunta, esperado in CASOS.items():
        direta = rec._busca(pergunta)
        nova = rec.reescrever(pergunta)                        # 1 chamada de LLM, compartilhada
        reescrita = rec._busca(nova)
        fusao = rec.fusao_com(pergunta, nova)
        for nome, res in (("direta", direta), ("reescrita", reescrita), ("fusão", fusao)):
            h1, h3 = acertos(res, esperado)
            linhas.append({"rep": rep, "pergunta": pergunta, "estratégia": nome, "hit@1": h1, "hit@3": h3, "reescrita": nova})
    print(f"repetição {rep} concluída")
ret = pd.DataFrame(linhas)

resumo = ret.groupby(["estratégia", "rep"])[["hit@1", "hit@3"]].mean().groupby("estratégia").agg(["mean", "min", "max"]).round(3)
resumo.loc[["direta", "reescrita", "fusão"]]

### B.1 Estabilidade por pergunta (em quantas das repetições o primeiro trecho estava certo)

In [ ]:
estab = ret.pivot_table(index="pergunta", columns="estratégia", values="hit@1", aggfunc="sum").astype(int)[["direta", "reescrita", "fusão"]]
estab.columns = [f"{c} (de {N_REP})" for c in estab.columns]
estab

In [ ]:
# Regressões: perguntas que a consulta direta acerta em hit@1 e a estratégia erra em 2 ou mais repetições
base_ok = {q for q, g in ret[ret["estratégia"] == "direta"].groupby("pergunta") if g["hit@1"].all()}
def regressoes(est):
    d = ret[ret["estratégia"] == est]
    return sorted(q for q in base_ok if (~d[d["pergunta"] == q]["hit@1"]).sum() >= 2)

m = ret.groupby("estratégia")[["hit@1", "hit@3"]].mean()
for est in ("reescrita", "fusão"):
    reg = regressoes(est)
    passa = bool(m.loc[est, "hit@1"] >= m.loc["direta", "hit@1"] and m.loc[est, "hit@3"] >= m.loc["direta", "hit@3"] and not reg)
    print(f"{est:10s} hit@1 {m.loc[est, 'hit@1']:.0%} (direta {m.loc['direta', 'hit@1']:.0%}) | hit@3 {m.loc[est, 'hit@3']:.0%} "
          f"(direta {m.loc['direta', 'hit@3']:.0%}) | regressões estáveis: {len(reg)} {reg}")
    print("  ->", "PASSA no critério" if passa else "NÃO passa no critério")
    if est == "fusão":
        adotar_fusao = passa
print()
print("DECISÃO (regra definida antes):", "ADOTAR a fusão como estratégia do agente" if adotar_fusao
      else "NÃO ADOTAR a fusão; manter a consulta direta e registrar o resultado")

## C. Conclusão (escrever a partir das saídas acima)
Preencha no relatório, com os **seus** números:
1. **Text-to-query:** quantos acertos a v1 e a v2 tiveram nas agregações e nos controles, e qual SQL errado apareceu (coluna `SQL` da tabela de erros). A decisão automática está impressa na seção A.
2. **Recuperação:** hit@1 e hit@3 médios (e o intervalo min a max entre repetições) da consulta direta, da reescrita e da fusão; quais perguntas continuam instáveis; e a decisão automática da seção B.
3. **Se algum ajuste não passou:** isso é um resultado válido. Registre que a técnica foi testada com critério prévio e rejeitada (ou que ficou inconclusiva por causa da variação do LLM).
4. **Limites:** 11 perguntas de recuperação e 5 de SQL, 3 repetições, sem teste estatístico, um único modelo (Gemini Flash Lite), embeddings lexicais (`hashing`).

**Para aplicar uma decisão de adoção:** SQL v2, trocar o padrão `versao="v1"` por `"v2"` em `criar_tool_sql` (`src/sql_agente.py`) e em `montar_ferramentas` (`src/tools.py`); fusão, usar `estrategia="fusao"` em `montar_ferramentas`.